# 02: YOLO11m Checkpoint Recovery, Dataset Extraction, Validation & Standby Retraining Pipeline
**Master Workspace:** `MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/08_COLAB_NOTEBOOKS`  
**Target Run Name (If Retraining Is Ever Triggered):** `yolo11m_baseline_retraining_2026`  
**Current Recovery Status:** **Condition 1 (`FOUND_AND_VERIFIED` / `READY_TO_USE`)**  
Both `yolo11m_primary_best.pt` (`1uFxCyHt2u2lqYmbEI9F3CkTEqbL_Wxbz`, `40,508,069 B`) and `yolo11m_baseline_best.pt` (`1fLveDJZPR5hh9X19VHJnXugw5Xb4MIWQ`, `40,508,069 B`) are present and verified in `04_MODEL_AND_CONFIGURATION/Checkpoints/`. Retraining is **not** required, and all training/evaluation cells below remain locked (`ENABLE_TRAINING = False`, `ENABLE_MODEL_EVALUATION = False`).


In [ ]:
# ==============================================================================
# CENTRALIZED GOOGLE DRIVE PATHS, EXECUTION LOCKS & SRC/ PACKAGE REGISTRATION
# ==============================================================================
ENABLE_ROBOFLOW_STAGING_DOWNLOAD = False
ENABLE_DATASET_REPLACEMENT = False
ENABLE_TRAINING = False
ENABLE_MODEL_EVALUATION = False
ENABLE_FULL_VIDEO_INFERENCE = False
ENABLE_TRACKER_REPLAY = False
ENABLE_MCDM_EVALUATION = False
ENABLE_IDENTITY_ABLATION = False
ENABLE_SEAT_PERTURBATION_SIMULATION = False
ENABLE_TEMPORAL_EXECUTION = False
ENABLE_MONTE_CARLO = False
ENABLE_GPU_EXPERIMENTS = False

import json
import logging
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s")
logger = logging.getLogger("YOLO11_IJITCS_SyncedColab")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

MASTER_ROOT = Path(
    "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026"
)
DATASET_ROOT = MASTER_ROOT / "03_DATASET_AUDIT" / "YOLO11_DATASET_UNZIPPED"
DATA_YAML_PATH = DATASET_ROOT / "data_colab.yaml"
REPO_ROOT = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "YOLO11_Repository"
VIDEO_PATH = (
    MASTER_ROOT
    / "02_ORIGINAL_RESEARCH_ARCHIVE"
    / "Original_Videos"
    / "video_part001.mp4"
)
CHECKPOINT_DIR = MASTER_ROOT / "04_MODEL_AND_CONFIGURATION" / "Checkpoints"
OUTPUT_DIR = MASTER_ROOT / "09_REVALIDATED_EXPERIMENT_OUTPUTS"
OUTPUT_PATH = OUTPUT_DIR

# Staging, Non-Destructive Backup, Archive & Supporting Artifact Paths
STAGING_DIR = MASTER_ROOT / "03_DATASET_AUDIT" / "_STAGING_ROBOFLOW_YOLOV11"
BACKUP_DATASET_DIR = MASTER_ROOT / "03_DATASET_AUDIT" / "_BACKUP_YOLO11_DATASET_UNZIPPED_PREV"
ZIP_ARCHIVE_PATH = MASTER_ROOT / "03_DATASET_AUDIT" / "bima_lppm2_.v2i.yolov11.zip"
DATASET_ZIP_PATH = ZIP_ARCHIVE_PATH
DATASET_PATH = DATASET_ROOT
DATA_ROBOFLOW_ORIG_YAML_PATH = DATASET_ROOT / "data_roboflow_original.yaml"
MODULES_DIR = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "Reconstructed_Research_Modules"

PRIMARY_BASELINE_CHECKPOINT = CHECKPOINT_DIR / "best.pt"
BASELINE_BEST_CHECKPOINT = CHECKPOINT_DIR / "yolo11m_baseline_best.pt"
FOCAL_BEST_CHECKPOINT = CHECKPOINT_DIR / "yolo11m_focal_best.pt"
BALANCED_BEST_CHECKPOINT = CHECKPOINT_DIR / "yolo11m_balanced_best.pt"
PRETRAINED_YOLO11M_PATH = CHECKPOINT_DIR / "yolo11m.pt"

if IN_COLAB and MASTER_ROOT.exists():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _resolve_existing(candidates: list[Path]) -> Path:
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]


TRACKING_VAL_DIR = MASTER_ROOT / "06_TRACKING_AND_IDENTITY_VALIDATION"
AHP_SAW_VAL_DIR = MASTER_ROOT / "07_AHP_SAW_AND_STATISTICAL_VALIDATION"
SUPPORTING_CONFIGS_DIR = TRACKING_VAL_DIR

STUDENT_SEATS_PATH = _resolve_existing([
    REPO_ROOT / "data" / "student_seats.csv",
    TRACKING_VAL_DIR / "student_seats.csv",
])
DETECTIONS_CSV_PATH = _resolve_existing([
    TRACKING_VAL_DIR / "detections.csv",
    OUTPUT_DIR / "Pipeline_Outputs" / "detections.csv",
])
TRACK_SUMMARY_CSV_PATH = _resolve_existing([
    TRACKING_VAL_DIR / "track_summary.csv",
    OUTPUT_DIR / "Pipeline_Outputs" / "track_summary.csv",
])
TRACK_TO_STUDENT_CSV_PATH = _resolve_existing([
    TRACKING_VAL_DIR / "track_to_student_summary.csv",
    OUTPUT_DIR / "Pipeline_Outputs" / "track_to_student_summary.csv",
])
STUDENT_AUDIT_CSV_PATH = _resolve_existing([
    TRACKING_VAL_DIR / "student_identity_audit.csv",
    OUTPUT_DIR / "Pipeline_Outputs" / "student_identity_audit.csv",
])
DURATION_MATRIX_CSV_PATH = _resolve_existing([
    AHP_SAW_VAL_DIR / "student_activity_matrix_duration.csv",
    TRACKING_VAL_DIR / "student_activity_matrix_duration.csv",
    OUTPUT_DIR / "Pipeline_Outputs" / "student_activity_matrix_duration.csv",
])
IDENTITY_ASSIGNMENTS_CSV_PATH = _resolve_existing([
    TRACKING_VAL_DIR / "identity_assignments.csv",
    OUTPUT_DIR / "Pipeline_Outputs" / "identity_assignments.csv",
])

# Register REPO_ROOT (for `from src.* import ...`) and MODULES_DIR on sys.path
for _p in (REPO_ROOT, MODULES_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# Invalidate cached `src.*` modules so re-running cells in Colab always loads the latest Drive files
import importlib
importlib.invalidate_caches()
for _m in list(sys.modules.keys()):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]

# Ensure roboflow and ultralytics are installed when running inside Google Colab
if IN_COLAB:
    import subprocess
    for _pkg in ("roboflow", "ultralytics"):
        try:
            __import__(_pkg)
        except ModuleNotFoundError:
            logger.info("Installing missing Colab dependency: %s ...", _pkg)
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

# Direct imports from canonical `src/` package (zero inline code duplication)
from src.dataset_manager import (
    ROBOFLOW_FORMAT,
    ROBOFLOW_PROJECT,
    ROBOFLOW_VERSION,
    ROBOFLOW_WORKSPACE,
    backup_and_replace_dataset,
    download_roboflow_to_staging,
    ensure_canonical_dataset_ready,
    generate_canonical_data_yamls,
    get_roboflow_api_key,
    inspect_ultralytics_labels_cache,
    validate_yolo_dataset_structure,
)
from src.detector_adapters_v2_aligned import (
    EXPECTED_CLASS_MAPPING,
    HISTORICAL_CHECKPOINT_RECONCILIATION,
    verify_yolo11_checkpoints,
)

logger.info("Registered REPO_ROOT on sys.path: %s", REPO_ROOT)
logger.info("Canonical DATASET_ROOT: %s | DATA_YAML_PATH: %s", DATASET_ROOT, DATA_YAML_PATH)


## Step 2: Idempotent Dataset Extraction & Integrity Validation (`1,373` Train / `393` Valid / `196` Test)


In [ ]:
# ==============================================================================
# ROBOFLOW STAGING DOWNLOAD, VALIDATION, SAFE REPLACEMENT & ZIP FALLBACK
# ==============================================================================
# Roboflow configuration:
#   Workspace : "universitas-negeri-semarang-giyam"
#   Project   : "sdn_ngaliyan-aulue"
#   Version   : 1
#   Format    : "yolov11"
# Set ROBOFLOW_API_KEY in Google Colab Secrets (userdata.get("ROBOFLOW_API_KEY"))
# or environment variable os.environ["ROBOFLOW_API_KEY"].

if ENABLE_ROBOFLOW_STAGING_DOWNLOAD:
    # Auto-install `roboflow` if not yet installed in this Colab runtime and reload src.dataset_manager
    if IN_COLAB:
        import subprocess
        try:
            import roboflow  # noqa: F401
        except ModuleNotFoundError:
            logger.info("Menginstal pustaka 'roboflow' secara otomatis...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "roboflow"])
    import importlib
    import src.dataset_manager as _dm
    importlib.reload(_dm)
    from src.dataset_manager import (
        backup_and_replace_dataset,
        download_roboflow_to_staging,
        get_roboflow_api_key,
    )

    # Reads ROBOFLOW_API_KEY from Colab Secrets / os.environ (or accepts a direct key string)
    rf_key = get_roboflow_api_key("ROBOFLOW_API_KEY")
    staging_report = download_roboflow_to_staging(
        api_key=rf_key,
        staging_dir=STAGING_DIR,
        workspace=ROBOFLOW_WORKSPACE,
        project_slug=ROBOFLOW_PROJECT,
        version_num=ROBOFLOW_VERSION,
        model_format=ROBOFLOW_FORMAT,
    )
    print(json.dumps(staging_report, indent=2))
    if staging_report.get("status") == "DOWNLOADED_AND_VALIDATED":
        replace_report = backup_and_replace_dataset(
            source_dir=Path(staging_report["resolved_yolo_root"]),
            canonical_dir=DATASET_ROOT,
            backup_dir=BACKUP_DATASET_DIR,
        )
        print(json.dumps(replace_report, indent=2))
elif IN_COLAB:
    dataset_status = ensure_canonical_dataset_ready(
        dataset_root=DATASET_ROOT,
        zip_archive_path=ZIP_ARCHIVE_PATH,
        staging_dir=STAGING_DIR,
        backup_dir=BACKUP_DATASET_DIR,
        prefer_roboflow_download=False,
    )
    print(json.dumps(dataset_status, indent=2))
else:
    logger.info(
        "[STANDBY] Dataset staging/extraction and data_colab.yaml auto-configuration are ready via src.dataset_manager."
    )


## Step 3: Standby Historical Baseline Retraining Pipeline (`yolo11m_baseline_retraining_2026`) (LOCKED)


In [ ]:
# ==============================================================================
# CHECKPOINT VERIFICATION & STANDBY HISTORICAL BASELINE RETRAINING (LOCKED)
# ==============================================================================
ckpt_verification = verify_yolo11_checkpoints(CHECKPOINT_DIR) if IN_COLAB else {"decision": "STANDBY", "selected_primary_checkpoint": str(PRIMARY_BASELINE_CHECKPOINT)}
logger.info("Checkpoint Verification: %s", ckpt_verification["decision"])

HISTORICAL_BASELINE_CONFIG = {
    "model": str(PRETRAINED_YOLO11M_PATH),
    "data": str(DATA_YAML_PATH),
    "epochs": 150,
    "patience": 30,
    "batch": 16,
    "imgsz": 640,
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "cos_lr": True,
    "close_mosaic": 10,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "mosaic": 1.0,
    "mixup": 0.0,
    "auto_augment": "randaugment",
    "erasing": 0.4,
    "seed": 42,
    "deterministic": True,
    "project": str(OUTPUT_DIR / "retraining_runs"),
    "name": "yolo11m_baseline_retraining_2026",
    "exist_ok": False,
}

if ckpt_verification["decision"] == "READY_TO_USE" and not ENABLE_TRAINING:
    logger.info("[CONDITION 1: READY_TO_USE] Baseline checkpoint verified (%s). Retraining skipped.", ckpt_verification["selected_primary_checkpoint"])
elif ENABLE_TRAINING:
    from ultralytics import YOLO
    model = YOLO(HISTORICAL_BASELINE_CONFIG["model"])
    train_kwargs = {k: v for k, v in HISTORICAL_BASELINE_CONFIG.items() if k != "model"}
    model.train(**train_kwargs)
else:
    logger.info("[LOCKED] Retraining pipeline prepared and locked (ENABLE_TRAINING = False).")


## Step 4: Separate Validation vs Held-Out Test Split Evaluation & Per-Class Export (LOCKED)


In [ ]:
# ==============================================================================
# SEPARATE VALIDATION ('val') VS HELD-OUT TEST ('test') EVALUATION (LOCKED)
# ==============================================================================
from src.detector_adapters_v2_aligned import evaluate_checkpoint_on_split
from src.evaluate_detector import evaluate_detector, evaluation_status

if not ENABLE_MODEL_EVALUATION:
    logger.info(
        "[LOCKED] Separate 'val' (393 images) and 'test' (196 images) evaluation via src.detector_adapters_v2_aligned "
        "and src.evaluate_detector is prepared and locked (ENABLE_MODEL_EVALUATION = False)."
    )
else:
    eval_out = OUTPUT_DIR / "checkpoint_evaluations_2026"
    eval_out.mkdir(parents=True, exist_ok=True)
    for split_target in ("val", "test"):
        res = evaluate_checkpoint_on_split(
            ckpt_path=PRIMARY_BASELINE_CHECKPOINT,
            data_yaml=DATA_YAML_PATH,
            split_name=split_target,
            out_dir=eval_out,
            enable_eval=ENABLE_MODEL_EVALUATION,
        )
        print(json.dumps(res, indent=2))
